In [1]:
import pyvisa
import logging

# Set root logger to only show INFO and above
logging.getLogger().setLevel(logging.WARNING)
# Every time plug and unlug, you need to restart
rm = pyvisa.ResourceManager()
print(rm.list_resources())
print(rm)
inst = rm.open_resource('GPIB0::16::INSTR') # Set this to even number
inst2 = rm.open_resource('GPIB0::22::INSTR')
print("Port 16 ", inst.query("*IDN?"))
print("Port 22" , inst2.query("*IDN?"))

('ASRL1::INSTR', 'GPIB0::16::INSTR', 'GPIB0::22::INSTR')
Resource Manager of Visa Library at C:\windows\system32\visa32.dll
Port 16  Agilent Technologies,B1500A,0,A.06.02.2023.0401

Port 22 Cascade Microtech,Velox,2448297,3.4.3.407


In [3]:
from autoprobe.cascade import CascadeAutoProbe
import pyvisa
import logging

# Clear all existing handlers
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Reconfigure root logger with your preferred settings
logging.basicConfig(
    level=logging.INFO,  # or DEBUG, WARNING, etc.
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)

# Silence all third-party libraries unless explicitly needed
for noisy_logger in ["pyvisa", "autoprobe", "urllib3", "chardet", "asyncio", "PIL"]:
    logging.getLogger(noisy_logger).setLevel(logging.CRITICAL)

autoprobe = CascadeAutoProbe(
    gpib_address="GPIB0::22::INSTR",
    data_folder="test/dec3",            # folder containing wafer.json, modules.json
    die_x=0,
    die_y=0,
    current_module="si_pmos_short_test_row_1",         # module name as in modules.json
    log=True,
    log_to_console=True
)

probe_temp = CascadeAutoProbe.load_die_modules("test/dec3", "modules.json")
instrument_cascade = rm.open_resource('GPIB0::22::INSTR')
instrument_cascade.write(f"ReadChuckPosition Y Z")
s = instrument_cascade.read() # read required to flush response
instrument_cascade.query("*OPC?")
chunks = s.split(" ") # format is like "0: 102500.321 102499.863 0.0"
x = float(chunks[1])
y = float(chunks[2])
z = float(chunks[3])
# logging.info(f"Initial chuck position: x0={x}, y0={y}, z0={z}")

autoprobe.move_chuck_relative(100, 0) # Move to the center of the chuck


2025-08-05 20:41:45,532 - INFO - Saving data at: test/dec3
2025-08-05 20:41:45,532 [MainThread  ] [INFO ]  Saving data at: test/dec3
2025-08-05 20:41:45,533 - INFO - Wafer size = 200 mm
2025-08-05 20:41:45,533 [MainThread  ] [INFO ]  Wafer size = 200 mm
2025-08-05 20:41:45,533 - INFO - Die size (x, y) = (11000, 26000)
2025-08-05 20:41:45,533 [MainThread  ] [INFO ]  Die size (x, y) = (11000, 26000)
2025-08-05 20:41:45,534 - INFO - Initial die wafer map coords: x=0, y=0
2025-08-05 20:41:45,534 [MainThread  ] [INFO ]  Initial die wafer map coords: x=0, y=0
2025-08-05 20:41:45,547 - INFO - Loaded 1426 die modules
2025-08-05 20:41:45,547 [MainThread  ] [INFO ]  Loaded 1426 die modules
2025-08-05 20:41:45,557 - INFO - Cascade Microtech,Velox,2448297,3.4.3.407
2025-08-05 20:41:45,557 [MainThread  ] [INFO ]  Cascade Microtech,Velox,2448297,3.4.3.407
2025-08-05 20:41:45,565 - INFO - Initial chuck position: x0=102022.09536, y0=102579.9937, z0=37289.45247
2025-08-05 20:41:45,565 [MainThread  ] [I